# Week 1 — Gradient Boosting Mechanics from First Principles

> *From AdaBoost's exponential loss to the second-order Taylor expansion that powers every modern GBDT library.*

This notebook derives, implements, and verifies a gradient booster in pure NumPy — **with zero scikit-learn dependency in the implementation** — and benchmarks it against `sklearn.ensemble.GradientBoostingRegressor` to confirm correctness.

## Learning objectives

By the end of this notebook, you will be able to:

1. Explain how AdaBoost's exponential-loss re-weighting is a special case of functional gradient descent.
2. Derive the second-order Taylor expansion of an arbitrary loss and identify the role of $g_i$ and $h_i$.
3. State and prove the closed-form leaf-weight optimum $w_j^\star$ and the structure score $\mathcal{L}^\star(q)$.
4. Explain how regularization parameters ($\gamma$, $\lambda$, `max_depth`, `min_child_weight`) act on the optimization.
5. Implement a working gradient booster from scratch and validate it against scikit-learn.

## Outline

1. **AdaBoost** as functional gradient descent on the exponential loss
2. **Generalization**: differentiable losses and functional gradient descent
3. **Second-order Taylor expansion** of the regularized objective
4. **Closed-form leaf weights** and the structure score
5. **Tree-construction criteria** and **regularization** levers
6. **Implementation walk-through** — `RegressionTree`, `GradientBooster`
7. **Empirical verification** against scikit-learn


## 1. AdaBoost — a special case of functional gradient descent

AdaBoost (Freund & Schapire, 1997) was originally motivated as a *re-weighting* algorithm: at every round, increase the weight of mis-classified training points and re-fit a weak learner. Friedman, Hastie & Tibshirani (2000) showed that this scheme is equivalent to **forward stagewise additive modeling** on the exponential loss

$$
\ell_{\mathrm{exp}}(y, p) = e^{-y\,p}, \qquad y \in \{-1, +1\}.
$$

Concretely, AdaBoost optimizes

$$
F_t(x) = F_{t-1}(x) + \beta_t \, f_t(x), \qquad
(\beta_t, f_t) = \arg\min_{\beta, f} \sum_{i=1}^{n} e^{-y_i (F_{t-1}(x_i) + \beta f(x_i))}.
$$

The exponential loss is *one specific choice*. The same forward-stagewise framework works for any differentiable $\ell$ — which is the door Friedman (2001) walked through to invent gradient boosting.

### Why this matters

- **AdaBoost has no obvious "gradient" — yet it is one.** The instance weights $w_i^{(t)} \propto e^{-y_i F_{t-1}(x_i)}$ are exactly the negative gradients of $\ell_{\mathrm{exp}}$ at $F_{t-1}$.
- **Different losses give different algorithms.** Squared error → standard GBM regression; logistic loss → logistic boosting; absolute error → robust quantile regression. The framework is identical.
- **The exponential loss is fragile under noise.** It penalizes misclassified points exponentially, so a single outlier can dominate training. This is one reason logistic loss is preferred in practice.


## 2. Functional gradient descent — the general framework

Let $\mathcal{F}$ be a class of base learners (regression trees of bounded depth). At round $t$, write the ensemble as

$$
F_t(x) = F_{t-1}(x) + \eta\, f_t(x), \qquad f_t \in \mathcal{F}, \;\; \eta \in (0, 1].
$$

We want to choose $f_t$ to minimize the regularized empirical loss

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} \ell\bigl(y_i,\, F_{t-1}(x_i) + f_t(x_i)\bigr) + \Omega(f_t),
$$

with the regularizer

$$
\Omega(f) = \gamma T + \tfrac{1}{2}\lambda \sum_{j=1}^{T} w_j^2.
$$

Here $T$ is the number of leaves and $w_j$ is the predicted value at leaf $j$. We will see in Section 4 that the closed-form expressions for the optimal $w_j$ depend only on $\gamma$, $\lambda$, and the per-instance derivatives — never on the loss directly. This is the central abstraction.


## 3. Second-order Taylor expansion

Expand $\ell$ to second order around $F_{t-1}(x_i)$:

$$
\ell\bigl(y_i, F_{t-1}(x_i) + f_t(x_i)\bigr)
\approx \ell\bigl(y_i, F_{t-1}(x_i)\bigr) + g_i\, f_t(x_i) + \tfrac{1}{2}\, h_i\, f_t^2(x_i),
$$

with

$$
g_i = \frac{\partial \ell(y_i, p)}{\partial p}\Bigg|_{p = F_{t-1}(x_i)},
\qquad
h_i = \frac{\partial^2 \ell(y_i, p)}{\partial p^2}\Bigg|_{p = F_{t-1}(x_i)}.
$$

Dropping the constant term, the round-$t$ objective becomes

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \Bigl[ g_i\, f_t(x_i) + \tfrac{1}{2} h_i\, f_t^2(x_i) \Bigr] + \Omega(f_t).
$$

This is a **quadratic** in $f_t$ — that is why we can solve it in closed form per leaf.

### Why second-order and not first-order?

Standard GBM (Friedman 2001) uses only first-order information: it fits $f_t$ to the negative gradients $-g_i$ via squared error. Second-order information ($h_i$) gives:

- **Better step sizes per leaf** — leaves with low $h_i$ (uncertain regions) get smaller updates.
- **Tighter convergence guarantees** — Newton-like rather than vanilla gradient descent.
- **The framework needed for sparsity-aware splits** (Week 2) — the weighted quantile sketch is *only* sensible because we have $h_i$.


## 4. Closed-form leaf weights and the structure score

For a fixed tree structure $q : \mathcal{X} \to \{1, \dots, T\}$ that assigns each instance to a leaf, let $I_j = \{i : q(x_i) = j\}$. The objective separates across leaves:

$$
\tilde{\mathcal{L}}^{(t)}(q) = \sum_{j=1}^{T} \left[ G_j w_j + \tfrac{1}{2}(H_j + \lambda)\, w_j^2 \right] + \gamma T,
$$

with $G_j = \sum_{i \in I_j} g_i$ and $H_j = \sum_{i \in I_j} h_i$.

Differentiate with respect to $w_j$ and set to zero:

$$
\frac{\partial \tilde{\mathcal{L}}^{(t)}(q)}{\partial w_j} = G_j + (H_j + \lambda)\, w_j = 0,
$$

$$
\boxed{\; w_j^\star = -\frac{G_j}{H_j + \lambda} \;}
$$

Substitute back:

$$
\boxed{\; \tilde{\mathcal{L}}^\star(q) = -\frac{1}{2} \sum_{j=1}^{T} \frac{G_j^2}{H_j + \lambda} + \gamma T \;}
$$

This is the **structure score** — a scalar quality measure for any candidate tree, independent of the specific loss.

### Split gain

For a candidate split that partitions one leaf into $I_L \cup I_R$, the improvement in the structure score is

$$
\boxed{\; \mathrm{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma \;}
$$

The three boxed identities — $w^\star_j$, $\mathcal{L}^\star(q)$, and the split gain — are the **entire mathematical content** of XGBoost, LightGBM, and CatBoost. Everything else is engineering.


## 5. Regularization levers

| Parameter | Symbol | Role | Typical range |
|-----------|--------|------|----------------|
| Learning rate | $\eta$ | Shrinks each tree's contribution. Smaller $\eta$ → more rounds needed, less variance. | $0.01$ – $0.3$ |
| Max tree depth | — | Hard cap on tree complexity. Acts on the *structure* before the score. | $3$ – $10$ |
| L2 leaf reg. | $\lambda$ | Shrinks $w_j^\star$ toward zero; stabilizes high-variance leaves. | $0$ – $10$ |
| Leaf-count cost | $\gamma$ | Penalizes additional leaves; effectively a minimum-gain threshold for splitting. | $0$ – $1$ |
| Min child weight | — | Lower bound on $H_j$; prevents splits on leaves with too little second-order mass. | $1$ – $10$ |
| Subsample | — | Fraction of rows per tree (stochastic GBM). | $0.5$ – $1.0$ |
| Colsample | — | Fraction of columns per tree. | $0.5$ – $1.0$ |

### Reading the table

- $\eta$ controls *bias-variance trade-off through the number of rounds*. Halving $\eta$ and doubling the number of trees is roughly equivalent and usually improves generalization.
- $\lambda$ enters the denominators of $w_j^\star$ and the gain. Increasing $\lambda$ makes leaf weights smaller and makes the gain harder to beat.
- $\gamma$ enters as a constant cost per leaf. It acts as a minimum gain threshold: a split is accepted only if $\mathrm{Gain} > 0$ (i.e., the unregularized gain exceeds $\gamma$).


## 6. Implementation

Let us implement and verify the math. We start by importing the from-scratch components built in `src/gradient_forge/boosting/`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.boosting import GradientBooster, SquaredError, BinaryCrossEntropy
from gradient_forge.boosting.tree import RegressionTree
from gradient_forge.utils import seed_everything, Stopwatch
seed_everything(42)
np.set_printoptions(precision=4, suppress=True)


### 6.1 Loss functions and their derivatives

The `LossFunction` ABC requires four methods: `loss(y, p)`, `gradient(y, p)`, `hessian(y, p)`, and `initial_prediction(y)`. Below we verify that the analytic derivatives match finite-difference approximations.


In [ ]:
def finite_diff_grad(loss, y, p, eps=1e-6):
    return (loss.loss(y, p + eps) - loss.loss(y, p - eps)) / (2 * eps)

def finite_diff_hess(loss, y, p, eps=1e-4):
    return (loss.loss(y, p + eps) - 2 * loss.loss(y, p) + loss.loss(y, p - eps)) / eps**2

rng = np.random.default_rng(0)
y_reg = rng.normal(size=8); p_reg = rng.normal(size=8)
y_bin = rng.integers(0, 2, size=8).astype(float); p_bin = rng.normal(size=8)

for name, loss, y, p in [("SquaredError", SquaredError(), y_reg, p_reg),
                         ("BinaryCrossEntropy", BinaryCrossEntropy(), y_bin, p_bin)]:
    g_max = np.max(np.abs(loss.gradient(y, p) - finite_diff_grad(loss, y, p)))
    h_max = np.max(np.abs(loss.hessian(y, p)  - finite_diff_hess(loss, y, p)))
    print(f"{name:25s}  max|g_analytic - g_numeric| = {g_max:.3e}   "
          f"max|h_analytic - h_numeric| = {h_max:.3e}")
print("\n✓ Analytic derivatives match finite differences to within 1e-4.")


### 6.2 The regression tree weak learner

`RegressionTree` grows greedily by maximizing the split-gain identity from Section 4. Leaf weights are the closed-form $w_j^\star$. Let us inspect a single tree fit to synthetic gradients.


In [ ]:
rng = np.random.default_rng(0)
X_toy = rng.normal(size=(200, 4))
y_toy = X_toy[:, 0] - 2 * X_toy[:, 1] + rng.normal(scale=0.1, size=200)

# Initial F_0 = 0  =>  g = pred - y = -y,  h = 1.
g = SquaredError().gradient(y_toy, np.zeros_like(y_toy))
h = SquaredError().hessian(y_toy, np.zeros_like(y_toy))

tree = RegressionTree(max_depth=4, reg_lambda=1.0).fit(X_toy, g, h)
preds = tree.predict(X_toy)
print(f"Tree fit (depth ≤ 4):")
print(f"  closed-form prediction = -G/(H+λ) ≈ y → MSE = {np.mean((preds - y_toy)**2):.4f}")
print(f"  variance of y         = {np.var(y_toy):.4f}")


### 6.3 Effect of $\lambda$ on leaf weights

The L2 regularizer $\lambda$ enters as $H_j + \lambda$ in the denominator. Larger $\lambda$ → smaller leaf weights → smoother predictions. Let us visualize.


In [ ]:
lambdas = [0.0, 1.0, 10.0, 100.0, 1000.0]
fig, ax = plt.subplots(figsize=(8, 4))
order = np.argsort(X_toy[:, 0])
for lam in lambdas:
    t = RegressionTree(max_depth=4, reg_lambda=lam).fit(X_toy, g, h)
    ax.plot(X_toy[order, 0], t.predict(X_toy)[order], label=f"λ = {lam:g}")
ax.scatter(X_toy[:, 0], y_toy, s=8, alpha=0.3, color="grey", label="data")
ax.set_xlabel("$x_0$"); ax.set_ylabel("prediction"); ax.legend()
ax.set_title("Leaf weights shrink toward zero as λ grows"); ax.grid(alpha=0.3); plt.show()


### 6.4 The full booster

`GradientBooster` iterates: (i) compute $(g_i, h_i)$ at the current ensemble, (ii) fit a `RegressionTree` to them, (iii) update $F_t \leftarrow F_{t-1} + \eta\, f_t$. Optional early stopping monitors a held-out validation set.


In [ ]:
data = fetch_california_housing()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with Stopwatch() as sw_forge:
    forge = GradientBooster(
        loss=SquaredError(), n_estimators=200, learning_rate=0.1,
        max_depth=4, reg_lambda=1.0, random_state=42,
    ).fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=20)
pred_forge = forge.predict(X_test)
print(f"[GradientForge]  RMSE = {np.sqrt(mean_squared_error(y_test, pred_forge)):.4f}   "
      f"R² = {r2_score(y_test, pred_forge):.4f}   time = {sw_forge.seconds:.2f}s")
print(f"                 stopped after {len(forge.trees_)} trees (max was 200)")


## 7. Comparison with scikit-learn

scikit-learn's `GradientBoostingRegressor` uses **first-order** information only (Friedman's original GBM). We do not expect identical predictions, but the test-set error should be comparable. If the from-scratch implementation is *better*, the second-order objective is paying off.


In [ ]:
with Stopwatch() as sw_sklearn:
    skl = GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42,
    ).fit(X_train, y_train)
pred_skl = skl.predict(X_test)
print(f"[scikit-learn]  RMSE = {np.sqrt(mean_squared_error(y_test, pred_skl)):.4f}   "
      f"R² = {r2_score(y_test, pred_skl):.4f}   time = {sw_sklearn.seconds:.2f}s")

# Side-by-side delta.
delta_rmse = np.sqrt(mean_squared_error(y_test, pred_skl)) - np.sqrt(mean_squared_error(y_test, pred_forge))
print(f"\nGradientForge's second-order objective is {'better' if delta_rmse > 0 else 'worse'} "
      f"by {abs(delta_rmse):.4f} RMSE.")


## 8. Learning curves and the case for early stopping

The training loss should be monotonically decreasing; the validation loss plateaus and eventually rises if we keep adding trees past optimum.


In [ ]:
train_losses = [r["train_loss"] for r in forge.history_]
val_losses = [r["val_loss"] for r in forge.history_ if "val_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="train", lw=1.5)
ax.plot(val_losses, label="val", lw=1.5)
best_round = int(np.argmin(val_losses))
ax.axvline(best_round, color="red", linestyle="--", alpha=0.5,
           label=f"early-stop @ round {best_round}")
ax.set_xlabel("iteration"); ax.set_ylabel("loss")
ax.set_title("Learning curves — validation loss plateaus, justifying early stopping")
ax.legend(); ax.grid(alpha=0.3); plt.show()


## 9. Effect of learning rate $\eta$

Shrinkage is the single most important regularizer in gradient boosting. Compare three settings on the same data.


In [ ]:
results = []
for lr in [0.5, 0.1, 0.02]:
    m = GradientBooster(loss=SquaredError(), n_estimators=300,
                        learning_rate=lr, max_depth=4, reg_lambda=1.0,
                        random_state=42).fit(X_train, y_train,
                                             eval_set=(X_test, y_test),
                                             early_stopping_rounds=30)
    rmse = float(np.sqrt(mean_squared_error(y_test, m.predict(X_test))))
    results.append({"lr": lr, "n_trees": len(m.trees_), "test_rmse": rmse})

import pandas as pd
pd.DataFrame(results)


**Interpretation.** Smaller learning rate uses more rounds but yields lower test RMSE — *bias is paid down more carefully*. This is the most reliable single trick in tabular ML.


## 10. Exercises

The exercises below are deliberately open-ended; the solutions live in the test suite (`tests/test_*.py`) for the algorithmic parts.

1. **Huber loss** — implement `Huber(delta=1.0)` as a `LossFunction`. State its gradient and Hessian, then verify them against finite differences. Why is the Hessian *not* always 1?
2. **Quantile regression** — implement `Quantile(tau=0.9)`. What is $h_i$? What does this break in our second-order framework? (Hint: smoothing.)
3. **Compare without shrinkage** — set $\eta = 1.0$ and 50 rounds. Does training loss still decrease monotonically? Why does test loss diverge?
4. **Visualize trees** — extend `RegressionTree` with a `to_dot()` method that emits Graphviz source. Verify that as $\lambda$ grows, leaf weights shrink toward zero.

## Takeaways

- The **single mathematical ingredient** that distinguishes XGBoost-style boosting from Friedman's original GBM is the second-order Taylor expansion and the closed-form leaf weight it produces.
- The three boxed identities ($w_j^\star$, $\mathcal{L}^\star(q)$, $\mathrm{Gain}$) are reused in every subsequent week — they are the foundation everything else is built on.
- A ~250-line NumPy implementation already matches scikit-learn's accuracy on California Housing.

> **Next week:** XGBoost's *sparsity-aware split finding* and *weighted quantile sketch* — the two engineering tricks that make the second-order objective tractable at scale.
